In [214]:
import gym
from gym import spaces
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from stable_baselines3 import PPO

In [221]:
class StockTradingEnv(gym.Env):
    def __init__(self, data):
        super(StockTradingEnv, self).__init__()
        self.data = data
        self.action_space = spaces.Discrete(3)  # 0: hold, 1: buy, 2: sell
        self.observation_space = spaces.Box(low=0, high=1, shape=(data.shape[1],))
        self.current_step = 0
        self.position = 0  # 0: no position, 1: holding stock
        self.total_profit = 0

    def reset(self):
        self.current_step = 0
        self.position = 0
        self.total_profit = 0
        return self.data[self.current_step, :]

    def step(self, action):
        reward = 0
        done = False

        if action == 1:  # buy
            if self.position == 0:
                self.position = 1
                self.buy_price = self.data[self.current_step][3]  # Assuming the 4th column is the closing price

        elif action == 2:  # sell
            if self.position == 1:
                self.position = 0
                reward = self.data[self.current_step][3] - self.buy_price
                self.total_profit += reward

        self.current_step += 1
        if self.current_step >= len(self.data) - 1:
            done = True

        return self.data[self.current_step, :], reward, done, {}

In [257]:
class MAMLAgent:
    def __init__(self, env, alpha=0.05, beta=0.0005, meta_iterations=200):
        self.env = env
        self.alpha = alpha  # inner loop learning rate
        self.beta = beta  # meta learning rate
        self.meta_iterations = meta_iterations
        self.model = self.build_model()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=self.beta)

    def build_model(self):
        inputs = tf.keras.Input(shape=(self.env.observation_space.shape[0],))
        x = tf.keras.layers.Dense(64, activation='relu')(inputs)
        x = tf.keras.layers.Dense(64, activation='relu')(x)
        outputs = tf.keras.layers.Dense(self.env.action_space.n, activation='softmax')(x)
        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        return model

    def adapt(self, trajectories):
        with tf.GradientTape() as tape:
            tape.watch(self.model.trainable_variables)
            loss = self.compute_loss(trajectories)
        grads = tape.gradient(loss, self.model.trainable_variables)
        adapted_weights = [w - self.alpha * tf.convert_to_tensor(g) for w, g in zip(self.model.trainable_variables, grads)]
        return adapted_weights

    def compute_loss(self, trajectories):
        total_loss = 0
        for trajectory in trajectories:
            obs, actions, rewards, _, _ = zip(*trajectory)
            obs = tf.convert_to_tensor(obs, dtype=tf.float32)
            actions = tf.convert_to_tensor(actions, dtype=tf.int32)
            rewards = tf.convert_to_tensor(rewards, dtype=tf.float32)
            action_probs = self.model(obs)
            action_mask = tf.one_hot(actions, depth=self.env.action_space.n)
            log_prob = tf.math.log(tf.reduce_sum(action_mask * action_probs, axis=-1))
            loss = -tf.reduce_mean(log_prob * rewards)
            total_loss += loss
        return total_loss / len(trajectories)

    def meta_learn(self, meta_batch):
        for _ in range(self.meta_iterations):
            meta_loss = 0
            for task in meta_batch:
                adapted_weights = self.adapt(task)
                self.model.set_weights(adapted_weights)
                with tf.GradientTape() as tape:
                    tape.watch(self.model.trainable_variables)
                    loss = self.compute_loss(task)
                grads = tape.gradient(loss, self.model.trainable_variables)
                self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
            meta_loss += loss

    def collect_trajectories(self, env, num_trajectories):
        trajectories = []
        for _ in range(num_trajectories):
            obs = env.reset()
            done = False
            trajectory = []
            while not done:
                obs = tf.convert_to_tensor(obs, dtype=tf.float32)
                action_probs = self.model(obs[None, :])
                action = np.random.choice(self.env.action_space.n, p=action_probs.numpy()[0])
                next_obs, reward, done, _ = env.step(action)
                trajectory.append((obs.numpy(), action, reward, next_obs, done))
                obs = next_obs
            trajectories.append(trajectory)
        return trajectories

In [258]:
class MAML_PPO:
    def __init__(self, env):
        self.env = env
        self.maml_agent = MAMLAgent(env)
        self.model = PPO('MlpPolicy', env, verbose=1)

    def train(self, total_timesteps):
        self.model.learn(total_timesteps=total_timesteps)
        meta_batch = self.collect_meta_batch()
        self.maml_agent.meta_learn(meta_batch)

    def collect_meta_batch(self):
        num_trajectories = 10
        meta_batch_size = 5
        meta_batch = [self.maml_agent.collect_trajectories(self.env, num_trajectories) for _ in range(meta_batch_size)]
        return meta_batch

    def save_model_weights(self, filepath):
        weights = self.model.get_weights()
        tf.keras.backend.set_learning_phase(0)
        tf.keras.models.save_model(self.model, filepath)
        tf.keras.backend.set_learning_phase(1)


In [256]:
def fetch_data():
    df = pd.read_csv('./datas/samsung.csv')
    data = df.drop(columns=['Date'])
    normalized_data = (data - data.min()) / (data.max() - data.min())
    return normalized_data.values

In [252]:
data = fetch_data()
env = StockTradingEnv(data)

In [253]:
maml_ppo = MAML_PPO(env)
maml_ppo.train(total_timesteps=10000)
maml_ppo.save_model_weights('./model/maml_ppo_model.h5')

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/Users/seokmin/Desktop/python/StockLearning/venv/lib/python3.12/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 244      |
|    ep_rew_mean     | 0.457    |
| time/              |          |
|    fps             | 8335     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 244         |
|    ep_rew_mean          | 0.459       |
| time/                   |             |
|    fps                  | 4257        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.006395421 |
|    clip_fraction        | 0.0128      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | 0.0723      |
|    learning_rate        | 0.

AttributeError: 'list' object has no attribute 'values'

In [ ]:
def evaluate_model(env, model):
    obs = env.reset()
    done = False
    while not done:
        obs = tf.convert_to_tensor(obs, dtype=tf.float32)
        action_probs = model(obs[None, :])
        action = np.argmax(action_probs.numpy()[0])
        obs, reward, done, _ = env.step(action)

In [ ]:
evaluate_model(env, maml_ppo.model)